In [ ]:
import pyvista as pv
import numpy as np
import netgen.meshing as ngm
from netgen.occ import *
import pickle

from ngsolve import *
from ngsolve.webgui import Draw
from ngsolve.bem import *

In [ ]:
mesh = pv.read('mesh.vtk')
points = mesh.points
trigs = mesh.faces.reshape(-1, 4)[:, 1:4]  # For triangles

In [ ]:
ngmesh = ngm.Mesh(dim=3)
ngmesh.AddPoints(points)
reg = ngmesh.AddRegion("surf", dim=2)
ngmesh.AddElements(2, reg, trigs)

In [ ]:
mesh = Mesh(ngmesh)
Draw (mesh);

In [ ]:
fes = SurfaceL2(mesh, order=1, complex=True)

u,v  = fes.TnT()
Id = BilinearForm(u*v*ds).Assemble()

f = 5000   # frequency
c = 343    # speed of sound
kappa = 2*pi*f / c

with TaskManager():
    # C = HelmholtzCombinedFieldOperator(fes, fes, kappa=kappa, intorder=4)
    C = HelmholtzCF(u*ds, kappa) * v*ds

lhs = 0.5 * Id.mat + C.mat

In [ ]:
rhs = Id.mat.CreateColVector()
rhs[fes.GetDofNrs(ElementId(BND,31048))[0]] = 1

In [ ]:
if True:
    gfu = GridFunction(fes)
    pre = BilinearForm(u*v*ds, diagonal=True).Assemble().mat.Inverse()
    with TaskManager():
        gfu.vec[:] = solvers.GMRes(A=lhs, b=rhs, pre=pre, maxsteps=50)

    outfile = open("sol.pkl", "wb")
    pickle.dump(gfu, outfile)

else:
    infile = open("sol.pkl", "rb")
    gfu = pickle.load(infile)

In [ ]:
Draw (gfu, min=-50, max=50);

In [ ]:
screen = WorkPlane(Axes( (0,0,0), Z, X)).RectangleC(1.5,1.5).Face()
vismesh = screen.GenerateMesh(maxh=0.02)
# Draw (vismesh);

In [ ]:
pot = C.GetPotential(gfu)
with TaskManager():
    pot.BuildLocalExpansion(vismesh.Boundaries(".*"))

In [ ]:
fes_screen = SurfaceL2(vismesh, order=2, complex=True)
uscat = GridFunction(fes_screen)

In [ ]:
with TaskManager():
    uscat.Set(pot, definedon=vismesh.Boundaries(".*"))

In [ ]:
Draw (uscat, min=-0.1, max=0.1, animate_complex=True);